In [91]:
import os

from dataclasses import dataclass
from typing import List, Dict, Any, Optional, Tuple, Callable

import torch
from torch import Tensor
from torch import quasirandom as t_qrand

import botorch
from botorch import fit as b_fit
from botorch import models as b_models
from botorch import acquisition as b_acq
from botorch import optim as b_optim
from botorch import test_functions as b_tfuncs
from botorch.utils import transforms as bu_transf
from botorch.posteriors import gpytorch as bp_gpytorch

import gpytorch
from gpytorch import likelihoods as g_lh
from gpytorch import constraints as g_constraints
from gpytorch import kernels as g_kern
from gpytorch import mlls as g_mlls

device: torch.device = torch.device("mps" if torch.mps.is_available() else "cpu")
tensor_dtype: torch.dtype = torch.float32

num_samples: int = 50
dim: int = 3
perturb_unconstrained_set_size: int = 500

max_iterations: int = 30

perturb_constraint: float = 0.1
lb: float = -20
ub: float = 20

confidence_level: float = 2.5
max_cholesky_size: float = float("inf")


In [92]:
@dataclass
class BOState():

    max_iterations: int
    current_iteration: int = 1

    def is_exceeded(
            self
            ) -> bool:
        return self.current_iteration > self.max_iterations
    def increment(
            self
            ) -> None:
        self.current_iteration += 1

In [93]:
ackley: b_tfuncs.Ackley = b_tfuncs.Ackley(
    dim=dim,
    negate=False,
).to(
    device=device,
    dtype=tensor_dtype
    )

ackley.bounds[0, :].fill_(lb)
ackley.bounds[1, :].fill_(ub)

def eval_ackley(x: Tensor):
    return ackley(x)

In [ ]:
def create_surrogate(
        X: Tensor,
        Y: Tensor,
        noise_interval: g_constraints.Interval = g_constraints.Interval(1e-6, 1e-4),
        matern_smoothness: float = 2.5,
) -> Tuple[b_models.SingleTaskGP, g_mlls.ExactMarginalLogLikelihood]:
    likelihood: g_lh.GaussianLikelihood = g_lh.GaussianLikelihood(
        noise_constraint=noise_interval,
    )
    kernel: g_kern.ScaleKernel = g_kern.ScaleKernel(
        base_kernel=g_kern.MaternKernel(
            nu=matern_smoothness,
        )
    )
    model: b_models.SingleTaskGP = b_models.SingleTaskGP(
        train_X=X,
        train_Y=Y,
        likelihood=likelihood,
        covar_module=kernel
    )
    mll: g_mlls.ExactMarginalLogLikelihood = g_mlls.ExactMarginalLogLikelihood(
        likelihood=likelihood,
        model=model,
    )
    return (model, mll)

def sample_candidate_positions(
        dim: int,
        batch_size: int,
        scramble: bool = True,
        center: bool = False,
        ) -> Tensor:
    sobol: t_qrand.SobolEngine = t_qrand.SobolEngine(
        dimension=dim,
        scramble=scramble
    )
    samples: Tensor = sobol.draw(n=batch_size).to(
        device=device,
        dtype=tensor_dtype
    ).requires_grad_(True)
    if not center:
        return samples

    return samples * 2 - 1

def get_ucb(
        model: b_models.SingleTaskGP,
        X: Tensor,
        beta: float = confidence_level,
        ) -> Tensor:
    posterior: bp_gpytorch.GPyTorchPosterior = model.posterior(X=X)
    return posterior.mean + beta * torch.sqrt(posterior.variance)

def get_lcb(
        model: b_models.SingleTaskGP,
        X: Tensor,
        beta: float = confidence_level,
        ) -> Tensor:
    posterior: bp_gpytorch.GPyTorchPosterior = model.posterior(X=X)
    return posterior.mean - beta * torch.sqrt(posterior.variance)


def generate_q_perturbations(
        q: int,
        tau: float,
) -> Tensor:
    samples: Tensor = sample_candidate_positions(
        dim=dim, 
        batch_size=q, 
        scramble=True,
        center=True
        ) * tau
    d_samples: Tensor = torch.cdist(
        samples, 
        torch.Tensor(
            [[0.0] * dim]
            ).to(device=device, dtype=tensor_dtype))
    mask: Tensor = (d_samples < tau).any(dim=1)
    masked: Tensor = samples[mask]
    return masked

def sample_q_perturbations(
        q: int,
        perturb_set: Tensor,
    ) -> Tensor:
    return torch.multinomial(
        input=perturb_set,
        num_samples=q,
        replacement=True
        )

def select_next_candidate(
        X: Tensor,
        perturb: Tensor,
    ) -> Tensor:
    p_match: Tensor = sample_q_perturbations(
        q=X.shape[0],
        perturb_set=perturb
    )
    acqf: Tensor 
    ...
    


In [95]:
x_safe: Tensor = torch.rand(
    size=[num_samples, dim],
    requires_grad=True,    
).to(device=device, dtype=tensor_dtype)
y_safe: Tensor = torch.tensor(
    [eval_ackley(x_safe[i, :]) for i in range(num_samples)]
).to(device=device, dtype=tensor_dtype)

state: BOState = BOState(
    max_iterations=max_iterations,
)

perturb_set: Tensor = generate_q_perturbations(
    q=perturb_unconstrained_set_size,
    tau=perturb_constraint,
)

while not state.is_exceeded():
    state.increment()
    model, mll = create_surrogate(
        X=x_safe.detach(),
        Y=y_safe.detach(),
    )

    with gpytorch.settings.max_cholesky_size(max_cholesky_size):
        b_fit.fit_gpytorch_mll(mll=mll)



    

torch.Size([273, 3])


BotorchTensorDimensionError: An explicit output dimension is required for targets. Expected Y with dimension 2 (got Y.dim()=1).